# QniNotebook チュートリアル

このノートブックでは、QURI Partsで作った小規模回路の中間状態を、Notebookから離れずに確認します。

## 目標
- QURI Parts の量子回路を作る
- `qni.show_circuit_and_state(circuit)` で回路と状態を表示する
- ステップ境界を選び、期待から外れたステップを特定する
- Pythonコードを修正して再実行する

> 初期デモは1〜8量子ビットの読み取り専用確認に限定します。GUI編集や `commit()`、大規模回路は扱いません。

## 準備

Qniの表示機能と、QURI Partsの量子回路を読み込みます。

In [7]:
from importlib import reload
from qni_jupyter import qni

# 同じカーネルに古いQniが残っていても、現在の実装を読み直す
qni.close()
qni = reload(qni)

from quri_parts.circuit import QuantumCircuit

## 1. まずは回路を作る

2 量子ビットの簡単な回路を作ります。ここでは H ゲートと CNOT ゲートを使います。

In [8]:
circuit = QuantumCircuit(2)
circuit.add_H_gate(0)
circuit.add_CNOT_gate(0, 1)

circuit

## 2. 回路と中間状態を表示する

`qni.show_circuit_and_state()` は、回路と状態ベクトル／Bloch表示を並べて表示します。左の回路で選択したステップ境界と、右の状態は常に同じ計算時点を表します。

In [9]:
qni.show_circuit_and_state(circuit)

QniViewer(url='http://127.0.0.1:46489/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22H%22%2C%22targets%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%5D%2C%22view%22%3A%22notebook%22%2C%22editable%22%3Afalse%2C%22active_step_index%22%3A1%2C%22qubit_count%22%3A2%2C%22backendUrl%22%3A%22http%3A//127.0.0.1%3A8000/backend.json%22%7D&height=280&qniCacheBust=1785742753375', height=280, width='100%', warnings=())

## 3. ステップごとの状態を確認する

量子回路は、ゲートを順番に適用していくことで状態が変化します。ここでは、回路全体ではなく「どのステップで状態が変わるか」を確認します。

Qni では、次のような流れで確認できます。
1. コードで回路を作る
2. Qni でその回路を見て理解する
3. 途中状態を見て、どこで状態が変わったかを確認する
4. 必要ならコードを修正して、再度確認する

上のQniで、回路の各ステップ境界を示す**縦棒**を選んでください。Hゲート直後の縦棒ではH適用後、CNOT直後の縦棒ではCNOT適用後までを計算した状態ベクトルが右側に表示されます。

ここでもう一度 `qni.show_circuit_and_state()` を実行する必要はありません。コードを変更したときだけ、上の表示セルを再実行します。古い出力が残っていても、それは以前の実行時点の読み取り専用スナップショットです。

## 4. 問題のあるステップを再現する

期待していなかった X ゲートがサブルーチンに入った状況を再現します。次の2セルを順に実行し、Xの直前と直後を比較してください。

In [4]:
# 既存の回路に X ゲートを追加し、回路そのものを変化させる
circuit.add_X_gate(1)
circuit

In [5]:
# 更新後の回路と状態ベクトルを表示する
qni.show_circuit_and_state(circuit)

QniViewer(url='http://127.0.0.1:46501/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22H%22%2C%22targets%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%5D%5D%2C%22view%22%3A%22notebook%22%2C%22editable%22%3Afalse%2C%22active_step_index%22%3A2%2C%22qubit_count%22%3A2%2C%22backendUrl%22%3A%22http%3A//127.0.0.1%3A8000/backend.json%22%7D&height=280&qniCacheBust=1785737791688', height=280, width='100%', warnings=())

X の直前ではBell状態、X の直後では別の状態になります。ステップ境界を順に選ぶことで、期待から外れた最初のステップがXだと特定できます。実際の利用では、このあとPythonコードを修正して表示セルを再実行します。

## 5. 回路だけを確認する（補助表示）

`qni.show_circuit()` は、状態ベクトルを付けずに回路だけを表示します。

In [6]:
qni.show_circuit(circuit)

QniViewer(url='http://127.0.0.1:46501/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22H%22%2C%22targets%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%5D%5D%2C%22view%22%3A%22circuit%22%2C%22editable%22%3Afalse%2C%22qubit_count%22%3A2%2C%22backendUrl%22%3A%22http%3A//127.0.0.1%3A8000/backend.json%22%7D&height=144&qniCacheBust=1785737797578', height=144, width=336, warnings=())

上のセルは、状態ベクトルを表示しない回路レビュー用の補助表示です。中間状態の調査には `show_circuit_and_state()` を使います。

## 初期デモの範囲

- 対応範囲は1〜8量子ビットの小規模回路です。
- 未対応ゲート、アンチコントロール、保持できないclassical bit mappingは、別の意味で表示せず例外で停止します。
- GUI編集、`commit()`、VQE/QAOAのパラメータ式、20〜32量子ビットの性能保証は初期デモに含みません。

これで、**回路を作る → 表示する → ステップ境界を比較する → 問題ステップを特定する → Pythonコードを直す**、という最小の確認フローは完了です。